In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
import time
from selenium.webdriver.common.by import By
import re
import os
import fitz
import requests
from datetime import datetime, timedelta
import shutil
from deep_translator import GoogleTranslator

In [ ]:
# translation


def translate_text(text, source='en', target='fr'):
    try:
        return GoogleTranslator(source=source, target=target).translate(text)
    except Exception as e:
        print("Translation error:", e)
        return text

In [ ]:
# normalise the date formats

def read_pdf_and_extract_dates(pdf_path):

    dates = []
    month_names_en = r"(?:january|february|march|april|may|june|july|august|september|october|november|december)"
    
    patterns = [
        r"\b\d{2}/\d{2}/\d{4}\b",                 
        r"\b\d{2}-\d{2}-\d{4}\b",                
        r"\b\d{4}-\d{2}-\d{2}\b",                 
        r"\b\d{1,2}\s+" + month_names_en + r"\s+\d{4}\b"
    ]

    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text()
            for pattern in patterns:
                matches = re.findall(pattern, text, flags=re.IGNORECASE)
                dates.extend(matches)

    return dates

In [ ]:
# parse the date

def parse_date(date_str):
    formats = [
        "%d/%m/%Y",
        "%d-%m-%Y",
        "%Y-%m-%d",
        "%d %B %Y",  
        "%d %b %Y"    
    ]
    
    for fmt in formats:
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            continue
    return None  

In [ ]:
driver = webdriver.Chrome()
driver.get("LINK") #add link
time.sleep(3)

In [ ]:
import re
from urllib.parse import urljoin, unquote, urlparse

BASE_URL = "https://www.icasa.org.za/bids-and-tenders/open-bids"
pdf_folder = "saved_files"
list_link_pdf = []
all_data = []
skipped = []        # Annexure J downloaded, but no date found anywhere
no_annexure = []    # tenders with no Annexure J PDF

# "10 July 2026", "24th of October 2025", "5 March 2025"
TEXT_DATE_RE = re.compile(
    r"\b(\d{1,2})(?:st|nd|rd|th)?\s+(?:of\s+)?"
    r"(January|February|March|April|May|June|July|August|September|October|November|December)"
    r"\s+(\d{4})\b",
    re.IGNORECASE,
)

def norm(s):
    """'A_nnexure-J-Bid-Document' -> 'annexurejbiddocument'"""
    return re.sub(r"[^a-z0-9]", "", s.lower())

def safe_name(name):
    return re.sub(r"[^\w.\-]+", "_", name)

def find_annexure_j_links(download_cell):
    """All PDFs in the Download cell whose link text or filename contains 'Annexure J'
    (the original and any erratum)."""
    found = []
    for a in download_cell.find_all("a", href=True):
        href = a["href"]
        filename = unquote(os.path.basename(urlparse(href).path))
        if "annexurej" in norm(a.get_text()) or "annexurej" in norm(filename):
            found.append((urljoin(BASE_URL, href), filename))
    return found

def dates_from_text(text):
    """Fallback: dates written in the title/description ('closing date extended to 10 July 2026')."""
    out = []
    for d, m, y in TEXT_DATE_RE.findall(text):
        parsed = parse_date(f"{d} {m} {y}")
        if parsed is not None:
            out.append(parsed)
    return out

# clear the saved_files folder
if os.path.exists(pdf_folder):
    shutil.rmtree(pdf_folder)
os.makedirs(pdf_folder, exist_ok=True)

counter = 0

while True:
    soup = BeautifulSoup(driver.page_source, "html.parser")
    table = soup.find("table", id="example")
    rows = table.find("tbody").find_all("tr")

    for row in rows:
        title = "?"
        try:
            cells = row.find_all("td")
            if len(cells) < 3:
                continue

            title_a = cells[0].find("a")
            title = title_a.get_text(strip=True) if title_a else cells[0].get_text(strip=True)
            detail_url = title_a["href"] if title_a and title_a.has_attr("href") else None
            description = cells[0].get_text(" ", strip=True)
            date = cells[1].get_text(strip=True)

            annexures = find_annexure_j_links(cells[2])
            if not annexures:
                no_annexure.append(title)
                continue

            parsed_dates = []
            main_link = annexures[0][0]

            for pdf_link, original_name in annexures:
                if pdf_link in list_link_pdf:
                    continue
                list_link_pdf.append(pdf_link)

                response = requests.get(pdf_link, stream=True, timeout=30)
                response.raise_for_status()
                counter += 1
                file_path = os.path.join(pdf_folder, f"{counter:04d}_{safe_name(original_name)}")
                with open(file_path, "wb") as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)

                dates = read_pdf_and_extract_dates(file_path)
                parsed_dates += [d for d in (parse_date(x) for x in dates if x) if d is not None]

            # fallback: dates in the tender's title / description
            if not parsed_dates:
                parsed_dates = dates_from_text(description)

            if not parsed_dates:
                skipped.append(main_link)
                continue

            expiration_date = max(parsed_dates)
            if expiration_date.date() >= datetime.now().date():
                all_data.append({
                    "title": title,
                    "publication_date": date,
                    "expiration_date": expiration_date.strftime("%Y-%m-%d"),
                    "link": main_link,
                    "detail_link": detail_url,
                    "country": "South Africa"
                })

        except Exception as e:
            print(f"Error for {title[:60]}: {e}")

    # pagination
    try:
        next_page = driver.find_element(By.ID, "example_next")
        if "disabled" not in next_page.get_attribute("class"):
            driver.execute_script("arguments[0].click();", next_page)
            time.sleep(1.5)
        else:
            break
    except Exception as e:
        print("Pagination error:", e)
        break

driver.quit()
print(f"{len(list_link_pdf)} Annexure J PDFs downloaded")
print(f"{len(no_annexure)} tenders without Annexure J")
print(f"{len(all_data)} valid notices found")
print(f"{len(skipped)} tenders with no readable date")

In [ ]:
# save the file

df= pd.DataFrame(all_data)
df.to_excel("../2_scraped_files/SouthAfrica.xlsx", index=False) # modify